# Use `wm_datasets` for video sampling

`wm_datasets` exposes clip and adjacent-frame datasets. These are for video pretraining workflows and have a different batch format from `world_model.data` trajectory windows. Provide a directory of `.mp4`, `.avi`, `.mov`, or `.mkv` files. The OpenCV loader works with the core project dependencies; `decord` is available through the optional `video` dependency.

## Why two video datasets?

A video clip contains temporal context: a sequence of frames can show how a scene changes. An adjacent-frame pair is a simpler self-supervised example for learning short-term visual changes. Neither format supplies the action-conditioned transitions needed by the world-model trainer in the trajectory notebook. Use these classes when working with raw videos, and `world_model.data` when you have synchronized robot trajectories.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from wm_datasets.videodataset import VideoDataset
from wm_datasets.pairdataset import PairDataset
from wm_datasets.datasample import VideoSample
from wm_datasets.datasets import discover_videos
from wm_datasets.utils.collate import video_collate, pair_collate

VIDEO_DIR = Path("/path/to/videos")  # edit to an existing directory
video_paths = discover_videos(str(VIDEO_DIR)) if VIDEO_DIR.is_dir() else []
print(f"Found {len(video_paths)} video files")

## Clips

`VideoDataset` samples a sequence of frames and returns `video` with shape `(C,T,H,W)`. The collator stacks these to `(B,C,T,H,W)`. Set `random_clip=False` for repeatable inspection.

In [ ]:
if video_paths:
    samples = [VideoSample(video_path=p) for p in video_paths]
    clips = VideoDataset(samples, num_frames=8, sampling_stride=2,
                         crop_size=128, random_clip=False, loader="opencv")
    clip_loader = DataLoader(clips, batch_size=2, collate_fn=video_collate)
    clip_batch = next(iter(clip_loader))
    print("Clips:", clip_batch["video"].shape)
    print("Frame indices:", clip_batch["frame_indices"][0])
else:
    print("Set VIDEO_DIR to inspect clips from your files.")

### Look at the sampled clip

`VideoDataset` normalizes each frame with ImageNet mean and standard deviation. The display cell reverses that normalization so the colors look natural. The index labels show the sampling stride.

In [ ]:
if video_paths:
    clip = clip_batch["video"][0]  # (C, T, H, W)
    mean = clip.new_tensor([0.485, 0.456, 0.406])[:, None, None]
    std = clip.new_tensor([0.229, 0.224, 0.225])[:, None, None]
    positions = [0, clip.shape[1] // 2, clip.shape[1] - 1]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for ax, pos in zip(axes, positions):
        frame = (clip[:, pos] * std + mean).clamp(0, 1)
        ax.imshow(frame.permute(1, 2, 0))
        ax.set_title(f"Frame {int(clip_batch['frame_indices'][0][pos])}")
        ax.axis("off")
    fig.suptitle("Temporal samples from one video")
    fig.tight_layout()
    plt.show()

## Adjacent frames

`PairDataset` samples two neighboring frames and returns `frame_t`, `frame_t1`, `indices`, and source path. With `random_pair=False`, it selects a deterministic middle pair.

In [ ]:
if video_paths:
    pairs = PairDataset(samples, random_pair=False, image_size=128,
                        loader="opencv")
    pair_loader = DataLoader(pairs, batch_size=2, collate_fn=pair_collate)
    pair_batch = next(iter(pair_loader))
    print("Frame pairs:", pair_batch["frame_t"].shape,
          pair_batch["frame_t1"].shape)
    print("Indices:", pair_batch["indices"])
else:
    print("Set VIDEO_DIR to inspect adjacent frames from your files.")

Use the [trajectory notebook](world_model_pipeline.ipynb) for synchronized RGB, depth, odometry, IMU, actions, and world model training. `wm_datasets` video batches do not supply those trajectory fields.